# Summary_Day12_online.ipynb  
## 튜닝 · 최적화 · 과적합 대응 · 인터넷 가능 버전 · CIFAR-10 기준

이번 12강은 **튜닝/최적화 및 과적합 대응**을 정리하는 강의다.

11강까지는 CNN 구조를 만들고 이미지 분류를 진행했다.  
이번 강의에서는 그 모델의 성능을 실제로 더 올리기 위해 무엇을 바꿔야 하는지 본다.

강의 흐름은 다음이다.

```text
신경망을 더 깊게 쌓기
→ 최적화 함수 선택
→ 과적합이 무엇인지 이해
→ Dropout으로 과도한 의존 막기
→ BatchNorm으로 학습 안정화
→ Data Augmentation으로 데이터 다양화
→ train/eval 모드 차이 이해
→ Softmax로 오답 확률 분석
```

이 파일은 **인터넷 가능 환경** 기준이다.  
`torchvision.datasets.CIFAR10(download=True)`를 사용한다.

> 필기 포인트:  
> 모델 성능은 모델 구조만으로 결정되지 않는다.  
> Optimizer, Dropout, BatchNorm, Data Augmentation 같은 튜닝 요소를 같이 조정해야 한다.

## 1. 전체 실습 목적

이번 실습의 목적은 다음이다.

1. 깊은 CNN 모델 `CNN_v2` 구조를 이해한다.
2. SGD, Momentum, Adam의 차이를 코드로 확인한다.
3. Dropout이 train mode와 eval mode에서 다르게 동작하는 것을 확인한다.
4. `CNN_v3`에서 Dropout을 어디에 배치하는지 본다.
5. `CNN_v4`에서 BatchNorm을 Conv 뒤, ReLU 앞에 넣는 이유를 이해한다.
6. `RandomHorizontalFlip`, `RandomErasing`으로 데이터 증강을 적용한다.
7. `net.train()`과 `net.eval()`의 차이를 확실히 잡는다.
8. Softmax 확률로 모델이 왜 틀렸는지 분석하는 흐름을 익힌다.

## 2. 라이브러리 준비

### 함수/모듈 사용법

```python
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset
```

- `torch`: Tensor 계산과 GPU 사용에 필요하다.
- `nn`: CNN, Dropout, BatchNorm, Loss를 만들 때 사용한다.
- `optim`: SGD, Adam 같은 최적화 함수를 사용할 때 필요하다.
- `transforms`: 이미지 전처리와 데이터 증강을 구성한다.
- `datasets`: CIFAR-10 데이터를 불러온다.
- `DataLoader`: mini-batch 단위로 데이터를 공급한다.
- `Subset`: 전체 데이터 중 일부만 골라 빠른 실습을 진행한다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset

from sklearn.metrics import classification_report, confusion_matrix


torch.manual_seed(123)
np.random.seed(123)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("PyTorch:", torch.__version__)

## 3. device 설정과 class 이름 준비

CIFAR-10은 10개 class를 가진 컬러 이미지 데이터다.

```text
plane, car, bird, cat, deer, dog, frog, horse, ship, truck
```

### 함수 사용법

```python
torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
```

- GPU가 있으면 `cuda:0`을 사용한다.
- GPU가 없으면 `cpu`를 사용한다.
- 모델과 Tensor는 반드시 같은 device에 있어야 한다.

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

n_output = len(classes)

print("device:", device)
print("class 수:", n_output)

## 4. Optimizer 기본: SGD 수동 업데이트

강의에서는 먼저 SGD가 파라미터를 어떻게 바꾸는지 간단히 확인했다.

기본 식은 다음이다.

```text
W = W - lr × W.grad
B = B - lr × B.grad
```

### 코드 사용법

```python
W.data -= lr * W.grad.data
```

- `W.grad`는 역전파로 계산된 gradient다.
- `lr`은 learning rate다.
- `.data`는 Tensor의 실제 값에 직접 접근한다.

> 실전에서는 직접 `.data`를 수정하기보다 `optimizer.step()`을 사용한다.  
> 여기서는 원리를 보여주기 위한 실습이다.

In [ ]:
W = torch.randn(3, 3, requires_grad=True)
B = torch.randn(3, requires_grad=True)

W.grad = torch.randn(3, 3)
B.grad = torch.randn(3)

lr = 0.001

print("업데이트 전 W 평균:", W.data.mean().item())
print("업데이트 전 B 평균:", B.data.mean().item())

W.data -= lr * W.grad.data
B.data -= lr * B.grad.data

print("업데이트 후 W 평균:", W.data.mean().item())
print("업데이트 후 B 평균:", B.data.mean().item())

## 5. Momentum과 Adam Optimizer 확인

### Momentum 사용법

```python
optimizer = optim.SGD(net_params, lr=lr, momentum=0.9)
```

- 기본 SGD에 관성 개념을 추가한다.
- 이전 업데이트 방향을 90% 정도 반영한다.
- 깊은 네트워크에서 초반 학습을 더 안정적으로 만들 수 있다.

### Adam 사용법

```python
optimizer = optim.Adam(net_params)
```

- Momentum 계열 방향 정보와 RMSprop 계열 보폭 조절을 함께 사용한다.
- 자동 학습률 조절 느낌이 있어 기본 선택지로 많이 쓰인다.

In [ ]:
net_params = [torch.randn(10, 1, requires_grad=True)]

optimizer_momentum = optim.SGD(net_params, lr=lr, momentum=0.9)
optimizer_adam = optim.Adam(net_params)

print("[Momentum]")
print(optimizer_momentum)

print("\n[Adam]")
print(optimizer_adam)

## 6. Dropout 동작 확인

Dropout은 학습 중 일부 뉴런을 무작위로 꺼서 과적합을 막는다.

### 함수 사용법

```python
dropout = nn.Dropout(0.5)
dropout.train()
dropout.eval()
```

- `Dropout(0.5)`: 50% 확률로 값을 0으로 만든다.
- `train()`: 학습 모드다. Dropout이 적용된다.
- `eval()`: 평가 모드다. Dropout이 적용되지 않는다.

> 강의 포인트:  
> train mode에서는 일부 값이 0이 되고, 살아남은 값은 보정되어 커진다.  
> eval mode에서는 입력이 그대로 통과한다.

In [ ]:
torch.manual_seed(123)

inputs = torch.randn(1, 10)

dropout = nn.Dropout(0.5)

dropout.train()
outputs_train = dropout(inputs)

dropout.eval()
outputs_eval = dropout(inputs)

print("Original Inputs:")
print(inputs)

print("\nOutputs in Train Mode:")
print(outputs_train)

print("\nOutputs in Eval Mode:")
print(outputs_eval)

## 7. 데이터 증강 Transform 정의

데이터 증강은 기존 이미지를 살짝 변형해 모델이 더 다양한 데이터를 보는 효과를 만든다.

강의에서 사용한 증강은 다음이다.

```text
RandomHorizontalFlip
RandomErasing
```

### 함수 사용법

```python
transforms.RandomHorizontalFlip(p=0.5)
```

- 50% 확률로 이미지를 좌우 반전한다.

```python
transforms.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3))
```

- 이미지 일부 사각형 영역을 무작위로 지운다.
- 모델이 특정 부분 하나에만 의존하지 않도록 돕는다.

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomErasing(
        p=0.5,
        scale=(0.02, 0.33),
        ratio=(0.3, 3.3),
        value=0,
        inplace=False
    )
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print(transform_train)

## 8. CIFAR-10 데이터 준비

인터넷 가능 환경에서는 CIFAR-10을 직접 다운로드한다.

### 함수 사용법

```python
datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
```

- `train=True`: 훈련 데이터다.
- `train=False`: 검증/테스트 데이터다.
- `download=True`: 데이터가 없으면 다운로드한다.
- `transform`: 이미지에 적용할 전처리다.

In [ ]:
data_root = "./data"

train_set_full = datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=transform_test
)

test_set_full = datasets.CIFAR10(
    root=data_root,
    train=False,
    download=True,
    transform=transform_test
)

train_set_aug_full = datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=transform_train
)

print("train:", len(train_set_full))
print("test:", len(test_set_full))

## 9. 빠른 실행용 Subset과 DataLoader

원본 강의는 50~100 epoch를 돌린다.  
실습 시간이 길 수 있으므로 Summary에서는 일부 데이터만 사용한다.

원본처럼 돌리려면 `train_size`, `test_size`, `num_epochs`를 늘리면 된다.

In [ ]:
train_size = 5000
test_size = 1000

train_indices = list(range(train_size))
test_indices = list(range(test_size))

train_set = Subset(train_set_full, train_indices)
test_set = Subset(test_set_full, test_indices)
train_set_aug = Subset(train_set_aug_full, train_indices)

batch_size = 100

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
train_loader_aug = DataLoader(train_set_aug, batch_size=batch_size, shuffle=True)

print("train batch:", len(train_loader))
print("test batch:", len(test_loader))
print("aug train batch:", len(train_loader_aug))

## 10. 이미지 출력 함수 만들기

CIFAR-10 이미지는 정규화되어 있어 그대로 출력하면 색이 이상할 수 있다.  
그래서 `(image * 0.5) + 0.5`로 역정규화해서 보여준다.

In [ ]:
def show_images_labels(loader, classes, net=None, device=None, n_show=20):
    images, labels = next(iter(loader))

    predicted = None

    if net is not None:
        net.eval()
        with torch.no_grad():
            inputs = images.to(device)
            outputs = net(inputs)
            predicted = torch.max(outputs, 1)[1].cpu()

    plt.figure(figsize=(10, 4))

    for i in range(min(n_show, len(images))):
        plt.subplot(2, 10, i + 1)

        img = images[i].permute(1, 2, 0).numpy()
        img = (img * 0.5) + 0.5
        img = np.clip(img, 0, 1)

        true_name = classes[labels[i].item()]

        if predicted is None:
            title = true_name
            color = "black"
        else:
            pred_name = classes[predicted[i].item()]
            title = f"{true_name}\n→ {pred_name}"
            color = "black" if predicted[i].item() == labels[i].item() else "red"

        plt.imshow(img)
        plt.title(title, fontsize=8, color=color)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_images_labels(test_loader, classes, None, None)

## 11. 공통 학습 함수 만들기

강의 원본에서는 `pythonlibs.torch_lib1`의 공통 함수를 불러온다.  
여기서는 인터넷이나 GitHub clone 없이 실행되도록 필요한 함수를 직접 만든다.

### 함수 역할

```text
torch_seed(): 난수 고정
fit(): 학습 루프
evaluate_history(): loss/accuracy 그래프 출력
collect_predictions(): 최종 예측 수집
```

In [ ]:
def torch_seed(seed=123):
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def fit(net, optimizer, criterion, num_epochs, train_loader, test_loader, device):
    history = []

    for epoch in range(num_epochs):
        net.train()

        train_loss = 0.0
        train_acc = 0.0
        n_train = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = net(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            predicted = torch.max(outputs, 1)[1]

            train_loss += loss.item() * labels.size(0)
            train_acc += (predicted == labels).sum().item()
            n_train += labels.size(0)

        net.eval()

        val_loss = 0.0
        val_acc = 0.0
        n_val = 0

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = net(inputs)
                loss = criterion(outputs, labels)
                predicted = torch.max(outputs, 1)[1]

                val_loss += loss.item() * labels.size(0)
                val_acc += (predicted == labels).sum().item()
                n_val += labels.size(0)

        history.append([
            epoch + 1,
            train_loss / n_train,
            train_acc / n_train,
            val_loss / n_val,
            val_acc / n_val
        ])

        print(
            f"epoch {epoch + 1} | "
            f"train_loss={history[-1][1]:.4f} | train_acc={history[-1][2]:.4f} | "
            f"val_loss={history[-1][3]:.4f} | val_acc={history[-1][4]:.4f}"
        )

    return np.array(history)


def evaluate_history(history, title="Learning Curve"):
    plt.plot(history[:, 0], history[:, 1], label="train loss")
    plt.plot(history[:, 0], history[:, 3], label="val loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title + " Loss")
    plt.legend()
    plt.show()

    plt.plot(history[:, 0], history[:, 2], label="train acc")
    plt.plot(history[:, 0], history[:, 4], label="val acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title(title + " Accuracy")
    plt.legend()
    plt.show()

    print("최종 검증 정확도:", history[-1, 4])

In [ ]:
def collect_predictions(net, loader, device):
    net.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = net(inputs)
            predicted = torch.max(outputs, 1)[1].cpu().numpy()

            y_true.extend(labels.numpy())
            y_pred.extend(predicted)

    return np.array(y_true), np.array(y_pred)

## 12. 깊은 CNN 모델 CNN_v2 만들기

강의의 기본 깊은 모델은 합성곱 6개와 풀링 3개로 구성된다.

설계 원칙은 다음이다.

```text
32채널 conv1-2 → pool
64채널 conv3-4 → pool
128채널 conv5-6 → pool
```

이미지 크기가 pooling으로 절반씩 줄어들 때, 채널 수는 2배씩 늘린다.  
공간 크기는 줄어도 더 많은 feature map으로 정보를 보완하는 구조다.

In [ ]:
class CNN_v2(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)

        self.conv5 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv6 = nn.Conv2d(128, 128, 3, padding=1)

        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d((2, 2))
        self.flatten = nn.Flatten()

        self.l1 = nn.Linear(4 * 4 * 128, 128)
        self.l2 = nn.Linear(128, num_classes)

        self.features = nn.Sequential(
            self.conv1, self.relu,
            self.conv2, self.relu,
            self.maxpool,

            self.conv3, self.relu,
            self.conv4, self.relu,
            self.maxpool,

            self.conv5, self.relu,
            self.conv6, self.relu,
            self.maxpool
        )

        self.classifier = nn.Sequential(
            self.l1,
            self.relu,
            self.l2
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

torch_seed()

net_v2 = CNN_v2(n_output).to(device)

dummy = torch.randn(2, 3, 32, 32).to(device)

with torch.no_grad():
    feature = net_v2.features(dummy)
    output = net_v2(dummy)

print(net_v2)
print("feature shape:", feature.shape)
print("output shape:", output.shape)

코드 해석:

- `padding=1`을 사용했기 때문에 Conv를 지나도 32×32 크기가 유지된다.
- `MaxPool2d(2,2)`를 지나면 공간 크기가 절반으로 줄어든다.
- 32×32 → 16×16 → 8×8 → 4×4가 된다.
- 마지막 feature map은 `[128, 4, 4]`다.
- Flatten하면 `128 × 4 × 4 = 2048`개 feature가 된다.

## 13. Optimizer 비교용 학습: SGD / Momentum / Adam

강의에서는 같은 `CNN_v2` 모델에 Optimizer만 바꿔 성능을 비교했다.

```text
SGD: 기본 최적화
Momentum: 이전 업데이트 방향을 반영
Adam: 방향 정보와 보폭 조절을 함께 사용
```

여기서는 빠른 실행을 위해 각 2 epoch만 학습한다.  
강의 원본처럼 비교하려면 epoch 수를 20~50으로 늘리면 된다.

In [ ]:
num_epochs = 2

histories = {}

for opt_name in ["SGD", "Momentum", "Adam"]:
    torch_seed()

    net = CNN_v2(n_output).to(device)
    criterion = nn.CrossEntropyLoss()

    if opt_name == "SGD":
        optimizer = optim.SGD(net.parameters(), lr=0.01)
    elif opt_name == "Momentum":
        optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.9)
    else:
        optimizer = optim.Adam(net.parameters())

    print("\n===", opt_name, "===")

    history = fit(
        net,
        optimizer,
        criterion,
        num_epochs,
        train_loader,
        test_loader,
        device
    )

    histories[opt_name] = history

In [ ]:
for opt_name, history in histories.items():
    plt.plot(history[:, 0], history[:, 4], label=opt_name)

plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.title("Optimizer Comparison")
plt.legend()
plt.show()

그래프 해석:

- Momentum과 Adam은 깊은 모델에서 SGD보다 빠르게 성능이 오를 수 있다.
- 짧은 epoch에서는 결과가 흔들릴 수 있다.
- 강의에서는 Momentum과 Adam이 더 빠른 수렴과 안정적 학습을 보였다.

## 14. CNN_v3: Dropout 추가 모델

Dropout 배치 전략은 다음이다.

```text
MaxPool 뒤에 Dropout 추가
분류기 Linear 사이에 Dropout 추가
깊어질수록 dropout 비율을 0.2 → 0.3 → 0.4로 증가
```

Dropout은 과적합을 줄이지만, 최적 성능까지 가는 데 더 많은 epoch가 필요할 수 있다.

In [ ]:
class CNN_v3(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)

        self.conv5 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv6 = nn.Conv2d(128, 128, 3, padding=1)

        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d((2, 2))
        self.flatten = nn.Flatten()

        self.dropout1 = nn.Dropout(0.2)
        self.dropout2 = nn.Dropout(0.3)
        self.dropout3 = nn.Dropout(0.4)

        self.l1 = nn.Linear(4 * 4 * 128, 128)
        self.l2 = nn.Linear(128, num_classes)

        self.features = nn.Sequential(
            self.conv1, self.relu,
            self.conv2, self.relu, self.maxpool,
            self.dropout1,

            self.conv3, self.relu,
            self.conv4, self.relu, self.maxpool,
            self.dropout2,

            self.conv5, self.relu,
            self.conv6, self.relu, self.maxpool,
            self.dropout3
        )

        self.classifier = nn.Sequential(
            self.l1,
            self.relu,
            self.dropout3,
            self.l2
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

net_v3 = CNN_v3(n_output).to(device)

print(net_v3)

## 15. CNN_v4: BatchNorm 추가 모델

BatchNorm은 데이터 분포를 안정화해 학습을 빠르고 안정적으로 만든다.

강의에서 강조한 위치는 다음이다.

```text
Conv → BatchNorm → ReLU
```

BatchNorm은 Conv가 만든 feature map의 분포를 먼저 안정화한 뒤 ReLU에 넣는다.

> 주의:  
> 채널 수가 같더라도 BatchNorm 인스턴스를 재사용하면 안 된다.  
> 각 위치마다 별도의 `nn.BatchNorm2d(...)`를 만들어야 한다.

In [ ]:
class CNN_v4(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)

        self.conv5 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv6 = nn.Conv2d(128, 128, 3, padding=1)

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(32)
        self.bn3 = nn.BatchNorm2d(64)
        self.bn4 = nn.BatchNorm2d(64)
        self.bn5 = nn.BatchNorm2d(128)
        self.bn6 = nn.BatchNorm2d(128)

        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d((2, 2))
        self.flatten = nn.Flatten()

        self.dropout1 = nn.Dropout(0.2)
        self.dropout2 = nn.Dropout(0.3)
        self.dropout3 = nn.Dropout(0.4)

        self.l1 = nn.Linear(4 * 4 * 128, 512)
        self.l2 = nn.Linear(512, num_classes)

        self.features = nn.Sequential(
            self.conv1, self.bn1, self.relu,
            self.conv2, self.bn2, self.relu, self.maxpool,
            self.dropout1,

            self.conv3, self.bn3, self.relu,
            self.conv4, self.bn4, self.relu, self.maxpool,
            self.dropout2,

            self.conv5, self.bn5, self.relu,
            self.conv6, self.bn6, self.relu, self.maxpool,
            self.dropout3
        )

        self.classifier = nn.Sequential(
            self.l1,
            self.relu,
            self.dropout3,
            self.l2
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

net_v4 = CNN_v4(n_output).to(device)

print(net_v4)

## 16. 증강 데이터로 CNN_v4 학습하기

강의에서는 `CNN_v4 + Adam + Data Augmentation` 조합으로 가장 높은 성능을 얻는 흐름을 보여줬다.

여기서는 빠른 실행을 위해 2 epoch만 기본으로 둔다.

In [ ]:
torch_seed()

net_final = CNN_v4(n_output).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net_final.parameters())

num_epochs = 2

history_final = fit(
    net_final,
    optimizer,
    criterion,
    num_epochs,
    train_loader_aug,
    test_loader,
    device
)

evaluate_history(history_final, title="CNN_v4 + Augmentation")

## 17. 최종 모델 예측 결과 확인

검증 이미지 일부에 대해 정답과 예측을 함께 출력한다.

In [ ]:
show_images_labels(test_loader, classes, net_final, device, n_show=20)

## 18. Softmax로 오답 확률 분석하기

모델의 출력은 logits다.  
class별 확률을 보려면 `torch.softmax(output, dim=1)`을 사용한다.

### 함수 사용법

```python
probs = torch.softmax(output, dim=1)
probs_np = probs.data.to("cpu").numpy()[0]
```

- `dim=1`: class 방향으로 softmax를 적용한다.
- `[0]`: batch 차원을 제거한다.
- 확률표를 보면 모델이 어떤 class와 헷갈렸는지 볼 수 있다.

In [ ]:
images, labels = next(iter(test_loader))

image = images[0]
label = labels[0]

plt.figure(figsize=(3, 3))

img = image.permute(1, 2, 0).numpy()
img = (img * 0.5) + 0.5
img = np.clip(img, 0, 1)

plt.imshow(img)
plt.title("true: " + classes[label.item()])
plt.axis("off")
plt.show()

net_final.eval()

with torch.no_grad():
    input_image = image.view(1, 3, 32, 32).to(device)
    output = net_final(input_image)
    probs = torch.softmax(output, dim=1)

probs_np = probs.data.to("cpu").numpy()[0]

names = np.array(classes)
values = np.array([f"{x:.4f}" for x in probs_np])

tbl = np.array([names, values]).T

print(tbl)
print("pred:", classes[int(np.argmax(probs_np))])

## 19. train mode와 eval mode 차이

PyTorch 모델에는 학습과 평가에서 다르게 동작하는 Layer가 있다.

대표적으로 다음 두 개다.

```text
Dropout
BatchNorm
```

### train mode

```python
net.train()
```

- Dropout이 일부 뉴런을 끈다.
- BatchNorm이 현재 mini-batch 통계로 정규화하고 `running_mean`, `running_var`를 업데이트한다.

### eval mode

```python
net.eval()
```

- Dropout을 끈다.
- BatchNorm은 훈련 중 누적된 running 통계를 사용한다.
- 예측 결과가 안정적으로 나온다.

## 20. BatchNorm 내부 동작 확인

BatchNorm은 상태가 있는 Layer다.

가지고 있는 값은 다음이다.

```text
running_mean
running_var
weight, gamma
bias, beta
```

훈련 중에는 현재 batch 평균/분산을 사용하고, 동시에 running 통계를 업데이트한다.  
평가 중에는 running 통계를 사용한다.

In [ ]:
torch.manual_seed(123)

bn_inputs = torch.randn(1, 1, 10)

i_mean = bn_inputs.mean()
i_var = bn_inputs.var(unbiased=True)
i_std = bn_inputs.std(unbiased=False)

bn = nn.BatchNorm1d(1)

print("입력 평균:", round(i_mean.item(), 4))
print("입력 표준편차:", round(i_std.item(), 4))
print("초기 running_mean:", bn.running_mean.data)
print("초기 running_var:", bn.running_var.data)

bn.train()
outputs_train = bn(bn_inputs)

print("\n훈련 출력:")
print(outputs_train.data.numpy().round(4))
print("훈련 후 running_mean:", bn.running_mean.data.numpy().round(4))
print("훈련 후 running_var:", bn.running_var.data.numpy().round(4))

bn.eval()
outputs_eval = bn(bn_inputs)

print("\n평가 출력:")
print(outputs_eval.data.numpy().round(4))

## 21. BatchNorm 재사용 실수

강의에서 주의한 핵심은 이것이다.

```text
BatchNorm은 같은 채널 수라도 같은 인스턴스를 재사용하면 안 된다.
```

잘못된 예시는 다음이다.

```python
self.conv1, self.bn1, self.relu
self.conv2, self.bn1, self.relu
```

`conv1` 뒤와 `conv2` 뒤가 같은 `bn1`을 공유하면 running 통계와 학습 파라미터가 섞인다.

올바른 방식은 다음이다.

```python
self.bn1 = nn.BatchNorm2d(32)
self.bn2 = nn.BatchNorm2d(32)

self.conv1, self.bn1, self.relu
self.conv2, self.bn2, self.relu
```

채널 수가 같아도 Layer 위치가 다르면 별도의 BatchNorm을 만든다.

## 22. 최종 평가 리포트

class별 성능을 확인한다.

In [ ]:
y_true, y_pred = collect_predictions(net_final, test_loader, device)

print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)

plt.imshow(cm)
plt.title("CIFAR-10 Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(range(10), classes, rotation=45, ha="right")
plt.yticks(range(10), classes)

for (i, j), value in np.ndenumerate(cm):
    if value > 0:
        plt.text(j, i, str(value), ha="center", va="center", fontsize=7)

plt.colorbar()
plt.tight_layout()
plt.show()

## 23. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `tuning` | 성능 개선을 위한 설정 조정 | optimizer, dropout, augmentation 등 |
| `overfitting` | 훈련 데이터에만 과하게 맞는 현상 | train loss 감소, val loss 증가 |
| `SGD` | 기본 최적화 함수 | `optim.SGD(...)` |
| `momentum` | 이전 이동 방향 반영 | `momentum=0.9` |
| `Adam` | 자동 보폭 조절 계열 Optimizer | `optim.Adam(...)` |
| `Dropout` | 일부 뉴런 비활성화 | `nn.Dropout(p)` |
| `BatchNorm2d` | 2D feature map 정규화 | `nn.BatchNorm2d(channels)` |
| `Data Augmentation` | 데이터 변형으로 다양성 증가 | transform에 추가 |
| `RandomHorizontalFlip` | 좌우 반전 | `p=0.5` |
| `RandomErasing` | 일부 영역 지우기 | 사각형 영역 삭제 |
| `train()` | 학습 모드 | dropout 적용, BN 통계 업데이트 |
| `eval()` | 평가 모드 | dropout 중단, BN running 통계 사용 |
| `running_mean` | BN 누적 평균 | eval에서 사용 |
| `running_var` | BN 누적 분산 | eval에서 사용 |
| `softmax` | logits를 확률로 변환 | `torch.softmax(output, dim=1)` |
| `padding=1` | Conv 후 크기 유지 | 3×3 Conv에서 자주 사용 |

## 24. 시험용 요약

```text
12강 핵심 = 깊은 CNN의 성능을 높이기 위한 튜닝 기법 정리
```

꼭 기억할 것:

- 깊은 모델은 더 복잡한 특징을 학습할 수 있다.
- 하지만 깊다고 무조건 좋은 것은 아니다.
- 깊은 모델은 학습이 느리거나 과적합이 생길 수 있다.
- Optimizer 선택은 성능과 수렴 속도에 큰 영향을 준다.
- SGD는 기본 Optimizer다.
- Momentum은 이전 업데이트 방향을 반영한다.
- Adam은 안정적이고 빠른 수렴을 기대할 수 있다.
- Overfitting은 train 성능은 좋아지는데 validation 성능이 나빠지는 현상이다.
- Dropout은 뉴런 일부를 무작위로 꺼서 과도한 의존을 막는다.
- Dropout은 train mode에서만 적용된다.
- BatchNorm은 데이터 분포를 안정화한다.
- BatchNorm은 보통 Conv 뒤, ReLU 앞에 둔다.
- BatchNorm 인스턴스는 각 위치마다 따로 만들어야 한다.
- Data Augmentation은 데이터를 변형해 일반화 성능을 높인다.
- RandomHorizontalFlip은 좌우 반전이다.
- RandomErasing은 이미지 일부를 지운다.
- `net.train()`과 `net.eval()`은 반드시 구분해야 한다.
- Softmax 확률을 보면 모델이 어떤 class와 헷갈렸는지 분석할 수 있다.